In [ ]:
import os
import copy
import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

config = {
    'window_size': 2048,
    'stride': 256,
    'batch_size': 64,
    'lr': 0.001,
    'epochs': 100,
    'num_classes': 10,
    'save_dir': './performance_results'
}

SNR_LIST = [-6, -4, -2, 0, 2, 4, 6]
SEED_LIST = [42, 123, 456, 789, 1010]

os.makedirs(config['save_dir'], exist_ok=True)


class ResidualShrinkageBlock(nn.Module):
    """残差收缩模块"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        self.fc = nn.Sequential(
            nn.Linear(out_channels, out_channels),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Linear(out_channels, out_channels),
            nn.Sigmoid()
        )
        
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )

    def forward(self, x):
        residual = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        x_abs = torch.abs(out)
        x_avg = self.global_pool(x_abs).view(x.size(0), -1)
        
        alpha = self.fc(x_avg).unsqueeze(2)
        threshold = alpha * x_avg.unsqueeze(2)
        
        out = torch.mul(torch.sign(out), torch.max(x_abs - threshold, torch.zeros_like(out)))
        
        if self.downsample is not None:
            residual = self.downsample(x)
            
        out += residual
        return self.relu(out)


class SpectrumCNN(nn.Module):
    """频域特征提取网络"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(32), 
            nn.ReLU(), 
            nn.MaxPool1d(2),
            
            nn.Conv1d(32, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(64), 
            nn.ReLU(), 
            nn.MaxPool1d(2),
            
            nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(128), 
            nn.ReLU()
        )

    def forward(self, x):
        fft_amp = torch.abs(torch.fft.rfft(x, dim=2))
        fft_amp = torch.log1p(fft_amp) 
        return self.net(fft_amp) 


class CrossDomainAttention(nn.Module):
    """跨域注意力机制"""
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, dropout=0.1)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x_t, x_p):
        q = x_t.permute(0, 2, 1)
        k = v = x_p.permute(0, 2, 1)
        attn_output, _ = self.mha(q, k, v)
        out = self.norm(q + attn_output)
        return out.permute(0, 2, 1)


class AdvancedPIMDNet(nn.Module):
    """主模型架构"""
    def __init__(self, num_classes):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=64, stride=4, padding=30, bias=False), 
            nn.BatchNorm1d(32), 
            nn.ReLU()
        )
        self.layer1 = ResidualShrinkageBlock(32, 32, stride=1)
        self.layer2 = ResidualShrinkageBlock(32, 64, stride=2)
        self.layer3 = ResidualShrinkageBlock(64, 128, stride=2)
        
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.path_phys = SpectrumCNN()
        self.cross_attn = CrossDomainAttention(embed_dim=128, num_heads=4)
        
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), 
            nn.BatchNorm1d(128), 
            nn.ReLU(), 
            nn.Dropout(0.4), 
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x_t = self.stem(x)
        x_t = self.layer1(x_t)
        x_t = self.layer2(x_t)
        x_t = self.layer3(x_t)
        
        x_p = self.path_phys(x) 
        x_t = self.cross_attn(x_t, x_p) 
        
        f_t = self.global_pool(x_t).flatten(1)
        f_p = self.global_pool(x_p).flatten(1)
        
        features = torch.cat([f_t, f_p], dim=1)
        return self.classifier(features)


class GearDataset(Dataset):
    """齿轮数据集加载器"""
    def __init__(self, samples_info, raw_data, labels_map, win_size, snr_db, mean, std):
        self.samples_info = samples_info
        self.raw_data = raw_data
        self.labels_map = labels_map 
        self.win_size = win_size
        self.snr_db = snr_db
        self.mean = mean
        self.std = std

    def __len__(self): 
        return len(self.samples_info)

    def _add_awgn(self, signal, snr_db):
        p_signal = np.mean(signal ** 2)
        if p_signal == 0: 
            return signal
        p_noise = p_signal / (10 ** (snr_db / 10.0))
        noise = np.random.normal(0, np.sqrt(p_noise), signal.shape)
        return signal + noise

    def __getitem__(self, idx):
        file_id, start = self.samples_info[idx]
        label = self.labels_map[file_id]
        
        segment = self.raw_data[file_id][start : start + self.win_size].copy()
        if self.snr_db is not None: 
            segment = self._add_awgn(segment, self.snr_db)
            
        segment = (segment - self.mean) / self.std
        
        x = torch.FloatTensor(segment).unsqueeze(0)
        y = torch.LongTensor([label]).squeeze()
        return x, y


def train_and_evaluate(snr, seed, raw_signals, labels_map):
    """单次实验的训练与评估流程"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(seed)

    all_pointers, all_labels = [], []
    for file_idx, signal in enumerate(raw_signals):
        label = labels_map[file_idx]
        n_samples = (len(signal) - config['window_size']) // config['stride'] + 1
        for i in range(n_samples):
            all_pointers.append((file_idx, i * config['stride']))
            all_labels.append(label)

    # 数据集划分 (20% Train, 20% Val, 60% Test)
    indices = np.arange(len(all_pointers))
    train_val_idx, test_idx, train_val_lbl, _ = train_test_split(
        indices, all_labels, test_size=0.60, random_state=seed, stratify=all_labels
    )
    train_idx, val_idx, _, _ = train_test_split(
        train_val_idx, train_val_lbl, test_size=0.50, random_state=seed, stratify=train_val_lbl
    )

    # 仅基于训练集计算均值和方差，避免数据泄露
    train_file_indices = set(all_pointers[i][0] for i in train_idx)
    all_train_points = np.concatenate([raw_signals[i] for i in train_file_indices])
    train_mean = np.mean(all_train_points)
    train_std = np.std(all_train_points) + 1e-6

    ds_kwargs = {
        'raw_data': raw_signals, 
        'labels_map': labels_map, 
        'win_size': config['window_size'], 
        'snr_db': snr, 
        'mean': train_mean, 
        'std': train_std
    }
    
    train_loader = DataLoader(
        GearDataset([all_pointers[i] for i in train_idx], **ds_kwargs), 
        batch_size=config['batch_size'], shuffle=True
    )
    val_loader = DataLoader(
        GearDataset([all_pointers[i] for i in val_idx], **ds_kwargs), 
        batch_size=config['batch_size']
    )
    test_loader = DataLoader(
        GearDataset([all_pointers[i] for i in test_idx], **ds_kwargs), 
        batch_size=config['batch_size']
    )

    model = AdvancedPIMDNet(num_classes=config['num_classes']).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

    best_val_acc = 0.0
    best_weights = None
    
    for epoch in range(config['epochs']):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            
        scheduler.step()
        
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds = model(x).argmax(dim=1)
                val_correct += (preds == y).sum().item()
                val_total += y.size(0)
        
        v_acc = 100 * val_correct / val_total
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_weights = copy.deepcopy(model.state_dict())

    # 加载最佳模型并在测试集上评估
    model.load_state_dict(best_weights)
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            preds = model(x).argmax(dim=1)
            test_correct += (preds == y).sum().item()
            test_total += y.size(0)
    
    return 100 * test_correct / test_total


if __name__ == '__main__':
    data_dir = r'F:\GearEccDataset\G1\T3\S600'
    files = [
        'N9_G1_E00_T3_S600.mat', 'N25_G1_E02_T3_S600.mat', 'N41_G1_E04_T3_S600.mat',
        'N57_G1_E06_T3_S600.mat', 'N73_G1_E08_T3_S600.mat', 'N89_G1_E10_T3_S600.mat',
        'N105_G1_E12_T3_S600.mat', 'N121_G1_E14_T3_S600.mat', 'N137_G1_E16_T3_S600.mat',
        'N153_G1_E18_T3_S600.mat'
    ]
    target_key = 'DataSensor03'
    
    raw_signals_list = []
    valid_labels_map = []

    print("Loading raw data...")
    for label_idx, fname in enumerate(files):
        fpath = os.path.join(data_dir, fname)
        with h5py.File(fpath, 'r') as f:
            data_seq = np.array(f[target_key]).reshape(-1).astype(np.float32)
            # 截断到统一长度
            data_seq = data_seq[:119808] if len(data_seq) >= 119808 else data_seq
            raw_signals_list.append(data_seq)
            valid_labels_map.append(label_idx)

    results = {}

    for snr in SNR_LIST:
        results[snr] = []
        print(f"\n--- Starting Evaluation for SNR: {snr} dB ---")
        
        for seed in SEED_LIST:
            acc = train_and_evaluate(snr, seed, raw_signals_list, valid_labels_map)
            results[snr].append(acc)
            
            # 保存单次结果
            log_file = os.path.join(config['save_dir'], f"result_snr{snr}_seed{seed}.txt")
            with open(log_file, 'w') as f:
                f.write(f"SNR: {snr} dB | Seed: {seed} | Accuracy: {acc:.2f}%\n")
            
            print(f"Seed {seed} completed. Accuracy: {acc:.2f}%")

    print("\n" + "="*40)
    print("Final Performance Summary")
    print("="*40)
    
    summary_file = os.path.join(config['save_dir'], "summary_table.txt")
    with open(summary_file, 'w') as f:
        header = "SNR\tMean ± Std\tAll Results\n"
        f.write(header)
        print(header.strip())
        
        for snr in SNR_LIST:
            accs = results[snr]
            mean_acc, std_acc = np.mean(accs), np.std(accs)
            
            # 格式化输出
            accs_str = ", ".join([f"{a:.2f}" for a in accs])
            row = f"{snr}dB\t{mean_acc:.2f} ± {std_acc:.2f}\t[{accs_str}]"
            
            print(row)
            f.write(row + "\n")
